In [1]:
pip install scipy

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import scipy
print(scipy.__version__)

1.16.3


In [4]:
import pandas as pd
import numpy as np
from scipy.optimize import linprog

# ==========================================
# PARAMETERS
# ==========================================

AVAILABLE_HOURS = 22

book_path = "C:/Users/Ex0164/Book1.xlsx"
daily_path = "C:/Users/Ex0164/Important codes/Child_for_13march_actual.xlsx"
matrix_path = "C:/Users/Ex0164/Important codes/compatibility_matrix.xlsx"

# ==========================================
# LOAD DATA
# ==========================================

hz_parts = pd.read_excel(book_path, sheet_name="HZ")
vt_parts = pd.read_excel(book_path, sheet_name="VT")

stats = pd.read_excel(book_path, sheet_name="Sheet2")
daily = pd.read_excel(daily_path, sheet_name="Sheet1")

hz_matrix = pd.read_excel(matrix_path, sheet_name="HZ_Matrix")

data = stats.merge(daily, left_on="Part", right_on="Material")

# ==========================================
# PRODUCTION RATE
# ==========================================

data["Rate"] = (3600 / data["Cycle time "].replace(0, np.nan)) * data["cavity"]

data = data[data["Rate"].notna()]

# ==========================================
# PARTS AND MACHINES
# ==========================================

parts = data["Material"].unique().tolist()

machines = hz_matrix.columns[1:].tolist()

# ==========================================
# LOOKUP DICTIONARIES
# ==========================================

# Clean inventory and demand before use
inventory = {k: (0 if pd.isna(v) else v) for k, v in inventory.items()}
demand = {k: (0 if pd.isna(v) else v) for k, v in demand.items()}


rate = dict(zip(data["Material"], data["Rate"]))

# ==========================================
# COMPATIBILITY
# ==========================================

compatibility = {}

for _, row in hz_matrix.iterrows():

    part = row["Part"]

    for m in machines:

        compatibility[(part, m)] = row[m]

# ==========================================
# CREATE VARIABLES
# ==========================================

variables = []
var_index = {}

k = 0

for p in parts:
    for m in machines:

        if compatibility.get((p, m), 0) == 1:

            variables.append((p, m))
            var_index[(p, m)] = k
            k += 1

n = len(variables)

# ==========================================
# OBJECTIVE
# ==========================================

c = np.ones(n) * 1

# ==========================================
# CONSTRAINT MATRICES
# ==========================================

A = []
b = []

# ------------------------------------------
# MACHINE CAPACITY
# ------------------------------------------

for m in machines:

    row = np.zeros(n)

    for p in parts:

        if (p, m) in var_index:

            idx = var_index[(p, m)]
            row[idx] = 1 / rate[p]

    A.append(row)
    b.append(AVAILABLE_HOURS)

# ------------------------------------------
# DEMAND CONSTRAINT
# ------------------------------------------

for p in parts:

    row = np.zeros(n)

    for m in machines:

        if (p, m) in var_index:

            idx = var_index[(p, m)]
            row[idx] = -1

    A.append(row)
    b.append(inventory.get(p, 0) - demand.get(p, 0))

# ==========================================
# SOLVE OPTIMIZATION
# ==========================================

result = linprog(
    c,
    A_ub=np.array(A),
    b_ub=np.array(b),
    bounds=(0, None),
    method="highs"
)

# ==========================================
# EXTRACT PLAN
# ==========================================

solution = result.x

plan = []

for (p, m), idx in var_index.items():

    qty = solution[idx]

    if qty > 1:

        plan.append({
            "Part": p,
            "Machine": m,
            "Qty": round(qty, 0),
            "Run_Hours": round(qty / rate[p], 2)
        })

plan_df = pd.DataFrame(plan)

plan_df.to_excel("Smart_APS_Plan_V1.xlsx", index=False)

print("Optimization Completed")

TypeError: 'NoneType' object is not subscriptable

In [5]:
print("Solver success:", result.success)
print("Solver message:", result.message)


Solver success: False
Solver message: The problem is infeasible. (HiGHS Status 8: model_status is Infeasible; primal_status is None)
